In [ ]:
# Compared to BPE and WordPiece, Unigram works in the other direction: it starts from a big vocabulary and removes tokens from it until it reaches the desired vocabulary size. There are several options to use to build that base vocabulary: we can take the most common substrings in pre-tokenized words, for instance, or apply BPE on the initial corpus with a large vocabulary size.

# At each step of the training, the Unigram algorithm computes a loss over the corpus given the current vocabulary. Then, for each symbol in the vocabulary, the algorithm computes how much the overall loss would increase if the symbol was removed, and looks for the symbols that would increase it the least. Those symbols have a lower effect on the overall loss over the corpus, so in a sense they are “less needed” and are the best candidates for removal.

# This is all a very costly operation, so we don’t just remove the single symbol associated with the lowest loss increase, but the
# p
# p (\(p\) being a hyperparameter you can control, usually 10 or 20) percent of the symbols associated with the lowest loss increase. This process is then repeated until the vocabulary has reached the desired size.


# https://huggingface.co/learn/llm-course/en/chapter6/7

In [ ]:
# Personalized note / version

# 1. Get all substring combinations for each word -> hug -> h u g, h ug, hu g, hug...

# 2. Count the frequency of this substring (e.g. "ug" is present in "hug","pug","hugs", )
        # ("h", 15) ("u", 36) ("g", 20) ("hu", 15) ("ug", 20) ("p", 17) ("pu", 17) ("n", 16)
        # ("un", 16) ("b", 4) ("bu", 4) ("s", 5) ("hug", 15) ("gs", 5) ("ugs", 5)

# 3. Compute the probability of all possible segmentation forming each word
        # ["p", "u", "g"] : 0.000389 -> P(p) * P(u) * P(g) = 5/210 * 36/210 * 20/210 = ...
        # ["p", "ug"] : 0.0022676
        # ["pu", "g"] : 0.0022676
    # We can take advantage using viterbi algorithm to find all segmentation forming a word

# 4. Get the loss by taking the initial best score for each word (e.g. "pug": ["pu", "g"] ), then experiment by removing one token at the time
    # see which token removal gives the least loss

# 5. Remove that token

In [1]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

1. Get word frequencies

In [3]:
from collections import defaultdict
def get_word_freqs(corpus):
    word_freqs = defaultdict(int)
    for text in corpus:
        words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
        new_words = [word for word, offset in words_with_offsets]
        for word in new_words:
            word_freqs[word] += 1
    
    return word_freqs
        

In [4]:
word_freqs = get_word_freqs(corpus)
word_freqs

defaultdict(int,
            {'▁This': 3,
             '▁is': 2,
             '▁the': 1,
             '▁Hugging': 1,
             '▁Face': 1,
             '▁Course.': 1,
             '▁chapter': 1,
             '▁about': 1,
             '▁tokenization.': 1,
             '▁section': 1,
             '▁shows': 1,
             '▁several': 1,
             '▁tokenizer': 1,
             '▁algorithms.': 1,
             '▁Hopefully,': 1,
             '▁you': 1,
             '▁will': 1,
             '▁be': 1,
             '▁able': 1,
             '▁to': 1,
             '▁understand': 1,
             '▁how': 1,
             '▁they': 1,
             '▁are': 1,
             '▁trained': 1,
             '▁and': 1,
             '▁generate': 1,
             '▁tokens.': 1})

2. Propose a candidate tokens, this is the GLOBAL version meant by unigram. So not form whole word, but from LARGE POOL of unverified tokens which we will trim further based on the probability-based on least reducing loss token. Get all substring, then sort by freq, then merge it with char-only pool

In [6]:
def init_token_freqs(word_freqs, max_vocab_size=300):
    char_freqs = defaultdict(int)
    subwords_freqs = defaultdict(int)

    for word, freq in word_freqs.items():
        # this is technically just a simple n^2 substring formation algorithm, so bruteforce-ly find all substring from a given word
        for i in range(len(word)):
            char_freqs[word[i]] += freq
            # loop through the subwords of length at least 2
            for j in range(i + 2, len(word) + 1):
                subwords_freqs[word[i:j]] += freq

    # sort subwords by frequency
    sorted_subwords = sorted(subwords_freqs.items(), key=lambda x: x[1], reverse=True)

    # however we wont take all subwords, we will take up to max_vocab - n_chars
    token_freqs = list(char_freqs.items()) + sorted_subwords[:max_vocab_size-len(char_freqs)]
    return {token: freq for token, freq in token_freqs}

In [7]:
token_freqs = init_token_freqs(word_freqs)
token_freqs

{'▁': 31,
 'T': 3,
 'h': 9,
 'i': 13,
 's': 13,
 't': 14,
 'e': 21,
 'H': 2,
 'u': 6,
 'g': 5,
 'n': 11,
 'F': 1,
 'a': 12,
 'c': 3,
 'C': 1,
 'o': 13,
 'r': 9,
 '.': 4,
 'p': 2,
 'b': 3,
 'k': 3,
 'z': 2,
 'w': 3,
 'v': 1,
 'l': 7,
 'm': 1,
 'f': 1,
 'y': 3,
 ',': 1,
 'd': 4,
 '▁t': 7,
 'is': 5,
 'er': 5,
 '▁a': 5,
 '▁to': 4,
 'to': 4,
 'en': 4,
 '▁T': 3,
 '▁Th': 3,
 '▁Thi': 3,
 '▁This': 3,
 'Th': 3,
 'Thi': 3,
 'This': 3,
 'hi': 3,
 'his': 3,
 'th': 3,
 'ou': 3,
 'se': 3,
 '▁tok': 3,
 '▁toke': 3,
 '▁token': 3,
 'tok': 3,
 'toke': 3,
 'token': 3,
 'ok': 3,
 'oke': 3,
 'oken': 3,
 'ke': 3,
 'ken': 3,
 '▁s': 3,
 'ra': 3,
 'nd': 3,
 '▁i': 2,
 '▁is': 2,
 '▁th': 2,
 '▁the': 2,
 'the': 2,
 'he': 2,
 '▁H': 2,
 'in': 2,
 'rs': 2,
 'te': 2,
 '▁ab': 2,
 'ab': 2,
 '▁tokeni': 2,
 '▁tokeniz': 2,
 'tokeni': 2,
 'tokeniz': 2,
 'okeni': 2,
 'okeniz': 2,
 'keni': 2,
 'keniz': 2,
 'eni': 2,
 'eniz': 2,
 'ni': 2,
 'niz': 2,
 'iz': 2,
 'at': 2,
 'ti': 2,
 'tio': 2,
 'tion': 2,
 'io': 2,
 'ion': 2,
 'on':

3. Prepare model for each -> model here technically just means a simple negative likelihood calculation

In [8]:
from math import log

total_sum = sum([freq for token, freq in token_freqs.items()])
model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

In [9]:
model

{'▁': 2.952892114877499,
 'T': 5.288267030694535,
 'h': 4.189654742026425,
 'i': 3.821929961901108,
 's': 3.821929961901108,
 't': 3.7478219897473863,
 'e': 3.342356881639222,
 'H': 5.6937321388027,
 'u': 4.59511985013459,
 'g': 4.777441406928545,
 'n': 3.9889840465642745,
 'F': 6.386879319362645,
 'a': 3.9019726695746444,
 'c': 5.288267030694535,
 'C': 6.386879319362645,
 'o': 3.821929961901108,
 'r': 4.189654742026425,
 '.': 5.000584958242754,
 'p': 5.6937321388027,
 'b': 5.288267030694535,
 'k': 5.288267030694535,
 'z': 5.6937321388027,
 'w': 5.288267030694535,
 'v': 6.386879319362645,
 'l': 4.440969170307332,
 'm': 6.386879319362645,
 'f': 6.386879319362645,
 'y': 5.288267030694535,
 ',': 6.386879319362645,
 'd': 5.000584958242754,
 '▁t': 4.440969170307332,
 'is': 4.777441406928545,
 'er': 4.777441406928545,
 '▁a': 4.777441406928545,
 '▁to': 5.000584958242754,
 'to': 5.000584958242754,
 'en': 5.000584958242754,
 '▁T': 5.288267030694535,
 '▁Th': 5.288267030694535,
 '▁Thi': 5.2882670

4. Viterbi algorithm to traverse every possible segmentation and get the best score

In [10]:
def encode_word(word, model):
    best_segmentations = [{"start": 0, "score": 1}] + [
        {"start": None, "score": None} for _ in range(len(word))
    ]
    for start_idx in range(len(word)):
        # This should be properly filled by the previous steps of the loop
        best_score_at_start = best_segmentations[start_idx]["score"]
        for end_idx in range(start_idx + 1, len(word) + 1):
            token = word[start_idx:end_idx]
            if token in model and best_score_at_start is not None:
                score = model[token] + best_score_at_start
                # If we have found a better segmentation ending at end_idx, we update
                if (
                    best_segmentations[end_idx]["score"] is None
                    or best_segmentations[end_idx]["score"] > score
                ):
                    best_segmentations[end_idx] = {"start": start_idx, "score": score}

    segmentation = best_segmentations[-1]
    if segmentation["score"] is None:
        # We did not find a tokenization of the word -> unknown
        return ["<unk>"], None

    score = segmentation["score"]
    start = segmentation["start"]
    end = len(word)
    tokens = []
    while start != 0:
        tokens.insert(0, word[start:end])
        next_start = best_segmentations[start]["start"]
        end = start
        start = next_start
    tokens.insert(0, word[start:end])
    return tokens, score

In [11]:
print(encode_word("Hopefully", model))
print(encode_word("This", model))

(['H', 'o', 'p', 'e', 'f', 'u', 'll', 'y'], 41.5157494601402)
(['This'], 6.288267030694535)


5. Compute loss now for each token calculated, obtain the one giving the least decrease

In [12]:
def compute_loss(model):
    loss = 0
    for word, freq in word_freqs.items():
        _, word_loss = encode_word(word, model)
        loss += freq * word_loss
    return loss

In [13]:
compute_loss(model)

413.10377642940875

In [14]:
import copy

def compute_scores(model):
    scores = {}
    model_loss = compute_loss(model)
    for token, score in model.items():
        # We always keep tokens of length 1
        if len(token) == 1:
            continue
        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)
        scores[token] = compute_loss(model_without_token) - model_loss
    return scores

6. Let's prune up until the X% tokens we wanna prune

In [15]:
percent_to_remove = 0.1
while len(model) > 100:
    scores = compute_scores(model)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1])
    # Remove percent_to_remove tokens with the lowest scores.
    for i in range(int(len(model) * percent_to_remove)):
        _ = token_freqs.pop(sorted_scores[i][0])

    total_sum = sum([freq for token, freq in token_freqs.items()])
    model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

Tokenize and enjoy using the constructed model / vocab

In [16]:
def tokenize(text, model):
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in words_with_offsets]
    encoded_words = [encode_word(word, model)[0] for word in pre_tokenized_text]
    return sum(encoded_words, [])

In [17]:
tokenize("This is the Hugging Face course.", model)

['▁This',
 '▁is',
 '▁the',
 '▁Hugging',
 '▁Face',
 '▁',
 'c',
 'ou',
 'r',
 's',
 'e',
 '.']